In [14]:
import os
import numpy as np
from tqdm import tqdm

KANJI_FEAT_DIR = "/app/notebooks/kanji_features"
all_feats = []

for fname in tqdm(os.listdir(KANJI_FEAT_DIR), desc="Collecting features"):
    if fname.endswith(".npy"):
        feat = np.load(os.path.join(KANJI_FEAT_DIR, fname))
        if feat.ndim == 2:
            all_feats.append(feat)

# Stack all patch embeddings
X = np.vstack(all_feats)  # shape: (total_patches, 512)
print("Total patches:", X.shape[0])


Total patches: 104664


In [15]:
from sklearn.cluster import KMeans

NUM_CLUSTERS = 30
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_


In [16]:
from sklearn.metrics import pairwise_distances_argmin_min

# Find closest patches to each centroid
closest_idxs, _ = pairwise_distances_argmin_min(centroids, X)

# Save or display representative patches
representative_patches = []
patch_count = 0
for fname in os.listdir(KANJI_FEAT_DIR):
    if fname.endswith(".npy"):
        patch_feat = np.load(os.path.join(KANJI_FEAT_DIR, fname))
        num_patches = patch_feat.shape[0]
        if patch_count + num_patches > max(closest_idxs):
            # Found the file containing one of the representatives
            img_name = fname.replace(".npy", ".png")
            from PIL import Image
            img = Image.open(os.path.join("/app/data/jouyou", img_name)).convert("L")
            for i, idx in enumerate(closest_idxs):
                if patch_count <= idx < patch_count + num_patches:
                    local_idx = idx - patch_count
                    x = (local_idx % 7) * 32
                    y = (local_idx // 7) * 32
                    patch = img.crop((x, y, x + 32, y + 32))
                    representative_patches.append(patch)
            if len(representative_patches) >= NUM_CLUSTERS:
                break
        patch_count += num_patches


In [18]:
import matplotlib.pyplot as plt

num_patches = len(representative_patches)
cols = 6
rows = int(np.ceil(num_patches / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))

for i in range(rows * cols):
    ax = axes.flat[i]
    if i < num_patches:
        ax.imshow(representative_patches[i], cmap='gray')
        ax.set_title(f"Cluster {i}", fontsize=6)
    ax.axis('off')  # always turn off axis

plt.tight_layout()
plt.savefig("/app/notebooks/patch_clusters.png", dpi=150)
